In [ ]:
# Husayn El Sharif
comments = """
Use environment sklearn_pytorch_env001.yml as it has scikit-learn which has k-means clustering
feature engineering steps for clustering AMI data
"""

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import os

# sklearn for k-means clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [16]:

output_file_path_30min = "../cleaned_data/SM Resampled Data BR2019 30min.csv"

if not os.path.exists(output_file_path_30min):
    print(f"File not found at: {output_file_path_30min}")
    print("Processing 3 minute data to create 30 minute data")

    # Load zipped CSV (already cleaned data)
    data_file_path = "../cleaned_data/SM Cleaned Data BR2019.csv.zip"
    df = pd.read_csv(data_file_path, compression="zip", low_memory=False)

    # --- Standardize timestamp column ---
    df = df.rename(columns={"x_Timestamp": "timestamp"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Drop rows with missing essentials
    df = df.dropna(subset=["timestamp", "meter"])

    # Sort (important for resampling)
    df = df.sort_values(["meter", "timestamp"])

    # --- Set index for resampling ---
    df = df.set_index("timestamp")

    # kWh should be summed, but voltage, current, frequency should be averaged
    agg_map = {
    "t_kWh": "sum",              # energy accumulates
    "z_Avg Voltage (Volt)": "mean",
    "z_Avg Current (Amp)": "mean",
    "y_Freq (Hz)": "mean",
    }

    # --- Resample to 30-minute intervals PER METER ---
    df_30min = (
        df
        .groupby("meter")
        .resample("30min")
        .agg(agg_map)             # use defined aggregation
        .dropna()
        .reset_index()
    )

    # save 30 minute resampled data
    df_30min.to_csv(output_file_path_30min, index=False)
    print(f"30-minute resampled data saved to: {output_file_path_30min}")

    
else:
    print(f"File already exists at: {output_file_path_30min}")
    print("Loading 30 minute data from file.")
    df_30min = pd.read_csv(output_file_path_30min, low_memory=False)
    df_30min["timestamp"] = pd.to_datetime(df_30min["timestamp"], errors="coerce")


# view first few rows of 30 minute data
df_30min.head()





File already exists at: ../cleaned_data/SM Resampled Data BR2019 30min.csv
Loading 30 minute data from file.


,meter,timestamp,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz)
0,BR02,2019-07-10 00:00:00,0.205,241.746,1.795,50.019
1,BR02,2019-07-10 00:30:00,0.200,242.452,1.741,50.029
2,BR02,2019-07-10 01:00:00,0.223,243.730,1.917,50.065
3,BR02,2019-07-10 01:30:00,0.223,244.752,1.893,50.029
4,BR02,2019-07-10 02:00:00,0.223,245.609,1.889,50.019


In [28]:
# Group by meter to compute features for clustering
meter_features_described = df_30min.groupby("meter").agg({
    "t_kWh": ["mean", "std", "max", "min"],
    "z_Avg Voltage (Volt)": ["mean", "std"],
    "z_Avg Current (Amp)": ["mean", "std"],
    "y_Freq (Hz)": ["mean", "std"],
})

meter_features_described

t_kWh                       z_Avg Voltage (Volt)             \
           mean       std    max  min                 mean        std   
meter                                                                   
BR02   0.095011  0.079462  0.727  0.0           236.810189  52.307234   
BR03   0.131199  0.094349  0.363  0.0           232.355339  54.595009   
BR04   0.207436  0.177531  1.608  0.0           238.086277  40.498318   
BR05   0.054964  0.042352  0.320  0.0           235.024342  47.687209   
BR06   0.309622  0.280565  2.394  0.0           242.096807  40.045844   
BR07   0.165193  0.236713  1.122  0.0           242.752350  40.705404   
BR08   0.167778  0.199669  0.951  0.0           241.586429  40.872001   
BR09   0.105769  0.066484  0.573  0.0           248.929986  37.021367   
BR10   0.135898  0.106841  0.690  0.0           235.553536  52.063779   
BR11   0.120355  0.125884  1.152  0.0           239.304040  32.586311   
BR12   0.234598  0.270213  1.263  0.0           241.227506  35.216185   
BR13   0.102025  0.092768  1.605  0.0           240.669688  36.284316   
BR14   0.083470  0.059842  0.587  0.0           243.449942  34.212021   
BR15   0.140382  0.109317  1.030  0.0           244.029821  35.259549   
BR16   0.017596  0.018476  0.195  0.0           229.839818  39.615483   
BR17   0.041994  0.067343  0.962  0.0           230.002999  39.506195   
BR18   0.285788  0.272525  1.685  0.0           230.729899  38.722160   
BR19   0.057417  0.070966  1.063  0.0           234.195380  39.082357   
BR20   0.031465  0.081663  1.274  0.0           231.240000  38.635057   
BR22   0.135917  0.107347  1.018  0.0           239.832945  35.689009   
BR23   0.294217  0.263444  1.218  0.0           246.958873  33.003944   
BR24   0.313496  0.319758  2.031  0.0           239.057987  33.437354   
BR26   0.321252  0.191699  0.952  0.0           225.990437  33.266507   
BR27   0.020589  0.020504  0.176  0.0           219.047782  44.788298   
BR28   0.156991  0.078121  0.486  0.0           226.700826  36.497219   
BR29   0.145471  0.113562  0.719  0.0           222.305627  33.931876   
BR30   0.050881  0.045729  0.298  0.0           235.423920  28.959536   
BR31   0.309973  0.332687  2.366  0.0           228.105627  35.913321   
BR32   0.153380  0.117030  0.726  0.0           211.217449  80.060545   
BR33   0.070245  0.063134  0.392  0.0           202.205850  76.121494   
BR34   0.087648  0.143234  1.281  0.0           217.889701  81.043879   
BR35   0.201635  0.177781  1.436  0.0           201.031842  81.367120   
BR36   0.179681  0.134131  0.759  0.0           202.731876  76.884548   
BR37   0.240552  0.172045  1.964  0.0           220.698664  82.760500   
BR38   0.209344  0.261161  1.863  0.0           211.677493  77.283872   
BR39   0.061709  0.064376  0.312  0.0           225.976553  83.379834   
BR42   0.007201  0.040070  0.753  0.0           204.757606  64.450544   
BR43   0.038600  0.034835  0.195  0.0           202.390267  65.899474   
BR44   0.028700  0.056830  0.726  0.0           228.217440  68.557309   
BR45   0.298051  0.303957  1.975  0.0           232.678107  68.677549   
BR46   0.050764  0.044306  0.432  0.0           226.830785  70.316678   
BR48   0.058539  0.046228  0.284  0.0           212.773785  64.193980   
BR49   0.156538  0.131340  1.118  0.0           253.313381  37.317556   
BR50   0.099032  0.105752  0.936  0.0           252.609679  36.803222   
BR51   0.204258  0.143934  1.276  0.0           251.673806  36.013058   
BR52   0.132746  0.138964  0.811  0.0           236.670836  42.111501   

      z_Avg Current (Amp)           y_Freq (Hz)             
                     mean       std        mean        std  
meter                                                       
BR02             0.913232  0.731615   47.154399  10.199785  
BR03             1.255726  0.883330   46.901033  10.714230  
BR04             1.978553  1.549518   48.256667   7.886357  
BR05             0.603433  0.490468   47.627997   9.406258  
BR06 

In [29]:
# sort 30 min data by meter and timestamp
df_30mins_sorted = df_30min.sort_values(["meter", "timestamp"])
df_30mins_sorted.head()

,meter,timestamp,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz)
0,BR02,2019-07-10 00:00:00,0.205,241.746,1.795,50.019
1,BR02,2019-07-10 00:30:00,0.200,242.452,1.741,50.029
2,BR02,2019-07-10 01:00:00,0.223,243.730,1.917,50.065
3,BR02,2019-07-10 01:30:00,0.223,244.752,1.893,50.029
4,BR02,2019-07-10 02:00:00,0.223,245.609,1.889,50.019


In [ ]:
# Types of features to engineer:
comment = """
For AMI clustering, consider four buckets:

A. Load behavior
B. Voltage stability
C. Frequency stability
D. Stress / dynamics
"""

def coeff_var(x):
    """Coefficient of variation with safety check."""
    mu = x.mean()
    coeff_of_var = (x.std() / mu) if mu != 0 else np.nan
    return coeff_of_var

def pct_outside_band(x, low, high):
    """Fraction of values outside tolerance band."""
    return ((x < low) | (x > high)).mean()

def ramp_stats(x):
    """Absolute ramp statistics. 
    On average, much does this meter's usage change (absolute value of the difference) 
    every 30 minutes?
    
    Interpretation:
    High ramp_mean → volatile / spiky usage
    Low ramp_mean → smooth / stable load profile

    Useful for:

    Customer segmentation
    Identifying unstable loads
    Clustering meters by “smooth vs spiky behavior”
    """
    dx = x.diff().abs()
    return pd.Series({
        "ramp_mean": dx.mean(),
        "ramp_p95": dx.quantile(0.95)
    })


In [ ]:
# Nominal voltage and frequency values (assumed constant across all meters)
V_NOM = df_30mins_sorted["z_Avg Voltage (Volt)"].median()
F_NOM = df_30mins_sorted["y_Freq (Hz)"].median()

V_LOW, V_HIGH = V_NOM * 0.95, V_NOM * 1.05
F_LOW, F_HIGH = F_NOM - 0.1, F_NOM + 0.1

V_NOM, F_NOM


(np.float64(245.12600000000003), np.float64(50.007))

In [34]:
# Aggregate to per-meter features

features = []

for meter_id, group_df in df_30mins_sorted.groupby("meter"):
    g = group_df.copy()
    g = g.sort_values("timestamp")

    kwh = g["t_kWh"]
    v = g["z_Avg Voltage (Volt)"]
    i = g["z_Avg Current (Amp)"]
    f = g["y_Freq (Hz)"]

    row = {
        "meter": meter_id,
        "n_records": len(g),

        # ── Load behavior ─────────────────────────
        "kwh_mean": kwh.mean(),
        "kwh_std": kwh.std(),
        "kwh_cv": coeff_var(kwh),
        "kwh_p95": kwh.quantile(0.95),
        "kwh_max": kwh.max(),

        # ── Voltage stability ─────────────────────
        "v_mean": v.mean(),
        "v_std": v.std(),
        "v_cv": coeff_var(v),
        "v_min": v.min(),
        "v_max": v.max(),
        "v_pct_outside_band": pct_outside_band(v, V_LOW, V_HIGH),

        # ── Frequency stability ───────────────────
        "f_mean": f.mean(),
        "f_std": f.std(),
        "f_min": f.min(),
        "f_max": f.max(),
        "f_pct_outside_band": pct_outside_band(f, F_LOW, F_HIGH),

        # ── Current / stress proxies ──────────────
        "i_mean": i.mean(),
        "i_p95": i.quantile(0.95),
        "i_max": i.max(),

        # ── Relationships ─────────────────────────
        "v_i_corr": v.corr(i),
        "kwh_v_corr": kwh.corr(v),
    }

    # Ramp features
    row.update(ramp_stats(kwh).add_prefix("kwh_"))
    row.update(ramp_stats(v).add_prefix("v_"))

    features.append(row)

meter_features = pd.DataFrame(features)
meter_features


,meter,n_records,kwh_mean,kwh_std,kwh_cv,kwh_p95,kwh_max,v_mean,v_std,v_cv,...,f_pct_outside_band,i_mean,i_p95,i_max,v_i_corr,kwh_v_corr,kwh_ramp_mean,kwh_ramp_p95,v_ramp_mean,v_ramp_p95
0,BR02,7248,0.095011,0.079462,0.836345,0.23765,0.727,236.810189,52.307234,0.220883,...,0.141142,0.913232,2.37300,6.222,0.158479,0.191054,0.034669,0.1590,11.642128,75.91300
1,BR03,6576,0.131199,0.094349,0.719128,0.29800,0.363,232.355339,54.595009,0.234963,...,0.148571,1.255726,2.79800,3.427,0.256470,0.273988,0.020789,0.0830,12.209766,77.25340
2,BR04,7584,0.207436,0.177531,0.855836,0.51000,1.608,238.086277,40.498318,0.170099,...,0.106672,1.978553,4.68925,13.457,0.046916,0.056466,0.073649,0.3060,8.811777,52.17140
3,BR05,5476,0.054964,0.042352,0.770536,0.13425,0.320,235.024342,47.687209,0.202903,...,0.124361,0.603433,1.64425,3.554,0.160096,0.189616,0.019955,0.0710,10.037751,68.72050
4,BR06,7488,0.309622,0.280565,0.906154,0.92500,2.394,242.096807,40.045844,0.165413,...,0.107772,2.822769,7.64095,18.638,0.129542,0.125267,0.154531,0.5947,8.716384,51.94560
5,BR07,7344,0.165193,0.236713,1.432944,0.71000,1.122,242.752350,40.705404,0.167683,...,0.108660,1.413811,5.69485,9.074,0.014258,0.020373,0.038458,0.2500,8.800522,52.57370
6,BR08,7584,0.167778,0.199669,1.190083,0.70200,0.951,241.586429,40.872001,0.169182,...,0.106145,1.625818,6.10685,8.675,0.010287,0.005341,0.046537,0.2580,8.900343,52.68090
7,BR09,4704,0.105769,0.066484,0.628572,0.25285,0.573,248.929986,37.021367,0.148722,...,0.077806,1.364484,2.60070,5.679,0.261258,0.218954,0.026017,0.1400,6.461246,26.86490
8,BR10,6816,0.135898,0.106841,0.786184,0.33200,0.690,235.553536,52.063779,0.221027,...,0.140258,1.302858,3.24325,7.467,0.155375,0.191972,0.044852,0.1860,11.402651,75.44280
9,BR11,6672,0.120355,0.125884,1.045945,0.24300,1.152,239.304040,32.586311,0.136171,...,0.098621,1.279336,2.36835,10.590,0.029536,0.033491,0.057171,0.1900,8.112489,49.15300


In [35]:
# Filter out meters with insufficient data
MIN_RECORDS = 200

meter_features = meter_features[
    meter_features["n_records"] >= MIN_RECORDS
].copy()

meter_features.shape


(46, 27)

In [ ]:
# Handle Missing Values 
num_cols = meter_features.select_dtypes(include=[np.number]).columns
meter_features[num_cols] = meter_features[num_cols].fillna(
    meter_features[num_cols].median()
)
meter_features.isnull().sum() 

meter                 0
n_records             0
kwh_mean              0
kwh_std               0
kwh_cv                0
kwh_p95               0
kwh_max               0
v_mean                0
v_std                 0
v_cv                  0
v_min                 0
v_max                 0
v_pct_outside_band    0
f_mean                0
f_std                 0
f_min                 0
f_max                 0
f_pct_outside_band    0
i_mean                0
i_p95                 0
i_max                 0
v_i_corr              0
kwh_v_corr            0
kwh_ramp_mean         0
kwh_ramp_p95          0
v_ramp_mean           0
v_ramp_p95            0
dtype: int64

In [37]:
# Save engineered features to CSV
output_features_path = "../cleaned_data/meter_engineered_features.csv"
meter_features.to_csv(output_features_path, index=False)
print(f"Engineered features saved to: {output_features_path}")

Engineered features saved to: ../cleaned_data/meter_engineered_features.csv
